# 01 — Análise Exploratória de Dados

Toda visualização tem um parágrafo de interpretação logo abaixo.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score

RANDOM_STATE = 42
CAMINHO_DADOS = Path('..') / 'data' / 'raw' / 'WineQT.csv'
TARGET = 'qualidade_do_vinho'
pd.set_option('display.max_columns', None)

## Compreensão do problema

A avaliação da qualidade de um vinho geralmente é feita por especialistas, por meio de provas sensoriais. Esse processo pode levar tempo, ter um custo elevado e apresentar diferenças de avaliação entre os profissionais. Ao mesmo tempo, durante a produção, já são realizadas análises físico-químicas em laboratório, como as de acidez, teor alcoólico e sulfatos.

Diante disso, a proposta deste trabalho é utilizar os resultados dessas análises para prever se um vinho pode ser considerado de alta qualidade. Dessa forma, a vinícola teria uma forma de avaliar os lotes antes da prova sensorial, identificar aqueles com potencial para uma linha premium e perceber possíveis problemas com antecedência. O uso de machine learning é adequado para esse objetivo, pois a qualidade do vinho pode estar relacionada à combinação de diferentes características, e não apenas a uma variável isolada.

Na base de dados, a coluna quality apresenta notas de 0 a 10. Inicialmente, analisamos a distribuição dessas notas e, posteriormente, transformamos a variável em duas classes, conforme o critério apresentado na seção 5 deste notebook (justificativa do limiar no notebook 02):

Alto: notas iguais ou superiores a 7.

Baixo/Médio: notas inferiores a 7.

In [ ]:
df_vinho = pd.read_csv(CAMINHO_DADOS)

## 1. Visão geral

### Fonte e dicionário de variáveis

A base é a WineQT, do Kaggle (Wine Quality Dataset).

As variáveis são:

- fixed acidity: acidez fixa, principalmente ácido tartárico (g/dm³)
- volatile acidity: acidez volátil, principalmente ácido acético (g/dm³)
- citric acid: ácido cítrico (g/dm³)
- residual sugar: açúcar que sobra depois da fermentação (g/dm³)
- chlorides: cloretos, quantidade de sal (g/dm³)
- free sulfur dioxide: dióxido de enxofre livre (mg/dm³)
- total sulfur dioxide: dióxido de enxofre total, livre + combinado (mg/dm³)
- density: densidade (g/cm³)
- pH: acidez em escala de pH
- sulphates: sulfatos (g/dm³)
- alcohol: teor alcoólico (% do volume)
- quality: nota de 0 a 10 dada por especialistas
- Id: numeração da linha, sem significado

In [ ]:
df_vinho.head(5)

In [ ]:
df_vinho.shape

In [ ]:
df_vinho.info()

In [ ]:
nulos = df_vinho.isnull().sum()
pd.DataFrame({'nulos': nulos, '%': (nulos / len(df_vinho) * 100).round(2)})

Nenhuma coluna tem valores faltantes.

In [ ]:
df_vinho.describe()

### Duplicatas

Olhando a base com a coluna Id não aparece nenhuma linha duplicada, porque o Id é diferente em todas. O Id é só a numeração da linha, então tiramos ele e verificamos de novo.

In [ ]:
df_vinho.duplicated().sum()

In [ ]:
df_vinho = df_vinho.drop('Id', axis=1)
df_vinho.duplicated().sum()

In [ ]:
df_vinho = df_vinho.drop_duplicates()
df_vinho.shape

Removemos as duplicatas para que o mesmo vinho não caia no treino e no teste ao mesmo tempo. Essa decisão também está registrada no notebook 02.

## 2. Distribuição das variáveis

### 2.1 Distribuição da nota (quality)

In [ ]:
df_vinho['quality'].value_counts()

In [ ]:
ax = sns.countplot(x='quality', data=df_vinho)
ax.bar_label(ax.containers[0])
plt.title('Quantidade de vinhos por nota')
plt.show()

A maior parte dos vinhos tem nota 5 ou 6 e não existe nenhum vinho com nota acima de 8. Notas muito baixas (3 e 4) e muito altas (8) são raras.

### 2.2 Distribuição das variáveis físico-químicas

In [ ]:
df_vinho.hist(figsize=(14, 10), bins=20)
plt.suptitle('Distribuição das variáveis')
plt.tight_layout()
plt.show()

In [ ]:
df_vinho.drop('quality', axis=1).skew().sort_values(ascending=False)

Chlorides, residual sugar, sulphates e os dióxidos de enxofre têm a distribuição bem puxada para a direita (poucos vinhos com valores muito altos). Density e pH são mais simétricas. As variáveis também estão em escalas diferentes.

## 3. Outliers

In [ ]:
colunas = df_vinho.columns.drop('quality')

plt.figure(figsize=(20,15))
for i, coluna in enumerate(colunas):
    plt.subplot(4, 3, i + 1)
    sns.boxplot(y=df_vinho[coluna])
    plt.title(coluna)
    plt.ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
for coluna in colunas:
    q1 = df_vinho[coluna].quantile(0.25)
    q3 = df_vinho[coluna].quantile(0.75)
    iqr = q3 - q1
    outliers = ((df_vinho[coluna] < q1 - 1.5 * iqr) | (df_vinho[coluna] > q3 + 1.5 * iqr)).sum()
    print(coluna, outliers)

In [ ]:
print((df_vinho < 0).sum().sum())
print(df_vinho[(df_vinho['pH'] < 2) | (df_vinho['pH'] > 5)].shape[0])
print(df_vinho[df_vinho['free sulfur dioxide'] > df_vinho['total sulfur dioxide']].shape[0])

Existem outliers principalmente em residual sugar, chlorides e sulphates, mas decidimos não remover. São valores possíveis para um vinho (mais sal, mais açúcar, mais sulfato) e não erro de medição. Também não encontramos valores inconsistentes: nada negativo, pH dentro do normal e o SO2 livre nunca é maior que o total.

## 4. Correlações

### 4.1 Variáveis por nota

In [ ]:
plt.figure(figsize=(20,15))
for i, coluna in enumerate(colunas):
    plt.subplot(4, 3, i + 1)
    sns.boxplot(x='quality', y=coluna, data=df_vinho, hue='quality', palette='hls', legend=False)
    plt.title(coluna)
    plt.ylabel('')
plt.tight_layout()
plt.show()

Nos boxplots por nota, alcohol, sulphates e citric acid sobem conforme a nota aumenta, e volatile acidity cai. Essa diferença fica mais clara a partir da nota 7. Nas outras variáveis as caixas das notas ficam bem parecidas.

### 4.2 Matriz de correlação

In [ ]:
correlation_matrix = df_vinho.corr().round(2)

fig, ax = plt.subplots(figsize=(10,10))
sns.heatmap(data=correlation_matrix, annot=True, linewidths=.5, ax=ax)
plt.show()

Análise das correlações

Ao analisar a relação entre as variáveis físico-químicas e a qualidade, observamos que o teor alcoólico (alcohol) tende a aumentar junto com a nota do vinho. Já a acidez volátil (volatile acidity) apresenta o comportamento contrário, com valores menores associados a notas mais altas. As variáveis sulphates e citric acid também apresentam uma relação positiva com a qualidade.

Também identificamos algumas relações entre as próprias características dos vinhos. A acidez fixa (fixed acidity) apresenta correlação negativa com o pH, o que faz sentido, já que uma maior acidez geralmente está associada a um pH menor. Além disso, a acidez fixa se relaciona com o ácido cítrico (citric acid) e com a densidade (density).

As variáveis free sulfur dioxide e total sulfur dioxide também apresentam uma correlação positiva, pois o dióxido de enxofre livre faz parte da quantidade total presente no vinho. Outra relação observada foi entre o álcool e a densidade, que apresentam correlação negativa.

Apesar dessas relações, nenhuma das correlações observadas entre as variáveis foi considerada alta o suficiente para justificar a exclusão de alguma característica antes do treinamento dos modelos.

## 5. Balanceamento de classes

#### Definição da variável alvo

Transformamos a nota em duas categorias usando o limiar de 7.

In [ ]:
def qualidade_do_vinho (qualidade):
  if qualidade < 7:
    return 0  #Qualidade Baixa/Média
  else:
    return 1 #Qualidade Alta

In [ ]:
df_vinho[TARGET] = df_vinho['quality'].apply(qualidade_do_vinho)

In [ ]:
df_vinho.head(5)

In [ ]:
df = df_vinho.drop('quality',axis=1)

In [ ]:
nomes = {0: 'Baixo/Médio', 1: 'Alto'}
df_grafico = df.copy()
df_grafico['qualidade_do_vinho'] = df_grafico['qualidade_do_vinho'].map(nomes)

In [ ]:
df_grafico['qualidade_do_vinho'].value_counts(normalize=True)

In [ ]:
ax = df_grafico['qualidade_do_vinho'].value_counts().plot(kind='bar', rot=0)
ax.bar_label(ax.containers[0])
plt.title('Quantidade de vinhos por categoria')
plt.xlabel('')
plt.show()

Só cerca de 13% dos vinhos são de qualidade alta, ou seja, as classes estão desbalanceadas. Por isso não dá para olhar só a acurácia: um modelo que prediz sempre "Baixo/Médio" já acertaria por volta de 86%.

## 6. Análise por categoria (Baixo/Médio x Alto)

In [ ]:
plt.figure(figsize=(20,15))
for i, coluna in enumerate(colunas):
    plt.subplot(4, 3, i + 1)
    sns.boxplot(x='qualidade_do_vinho', y=coluna, data=df_grafico, hue='qualidade_do_vinho', palette='hls', legend=False, order=['Baixo/Médio', 'Alto'])
    plt.title(coluna)
    plt.xlabel('')
    plt.ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
df_grafico.groupby('qualidade_do_vinho')[colunas].median().T

Os boxplots e as medianas confirmam a análise por nota: os vinhos Alto têm mais alcohol, menos volatile acidity, mais sulphates e mais citric acid. Nas outras variáveis as caixas das duas categorias se sobrepõem bastante.

In [ ]:
sns.pairplot(df_grafico, vars=colunas, hue='qualidade_do_vinho')

No pairplot, nenhuma variável sozinha separa as duas categorias. Os vinhos Alto ficam mais concentrados na região de álcool alto e acidez volátil baixa, mas ainda misturados com vinhos Baixo/Médio. Por isso faz sentido usar modelos que combinam várias variáveis ao mesmo tempo.

In [ ]:
correlacao = df.corr(method='spearman')['qualidade_do_vinho'].drop('qualidade_do_vinho').sort_values()
ax = correlacao.plot(kind='barh', figsize=(8,5))
ax.bar_label(ax.containers[0], fmt='%.2f')
plt.title('Correlação (Spearman) com qualidade Alta')
plt.xlim(-0.45, 0.45)
plt.show()

Aqui usamos a correlação de Spearman porque várias variáveis têm outliers e distribuição assimétrica, e ela é menos afetada por isso.

Correlações com a qualidade Alta e sua consequência nos vinhos:
- alcohol (positiva): vinhos com fermentação completa e uvas mais maduras têm mais álcool, o que dá mais corpo ao vinho.
- sulphates (positiva): o sulfato funciona como conservante e antioxidante, ajudando a manter o aroma.
- citric acid (positiva): dá frescor ao vinho.
- volatile acidity (negativa): é principalmente ácido acético. Em excesso deixa o vinho com cheiro de vinagre.
- chlorides (negativa): muito sal deixa o vinho desequilibrado.
- density (negativa): está ligada ao álcool. O álcool é menos denso que a água, então quanto mais álcool, menor a densidade.
- total sulfur dioxide (negativa): SO2 em excesso aparece no aroma.

## 7. Síntese da EDA

Depois de remover as duplicatas, a base tem 1.018 vinhos, sem valores faltantes. As notas se concentram em 5 e 6, e notas muito baixas ou muito altas são raras.

Separando os vinhos em Alto (nota igual ou superior a 7) e Baixo/Médio, só cerca de 13% são Alto. Por isso a avaliação dos modelos precisa olhar para precisão, recall e F1 da classe Alto, e não só para a acurácia.

As variáveis que mais diferenciam os vinhos Alto são alcohol (mais alto), volatile acidity (mais baixa), sulphates e citric acid (mais altos). Isso aparece de forma consistente nos boxplots por nota, nos boxplots por categoria, nas medianas e na correlação de Spearman.

Nenhuma variável separa as categorias sozinha: no pairplot os vinhos Alto ficam concentrados na região de álcool alto e acidez volátil baixa, mas ainda misturados com vinhos Baixo/Médio. Por isso faz sentido usar modelos que combinam várias variáveis ao mesmo tempo.

Os outliers foram mantidos por serem valores possíveis, e as variáveis estão em escalas bem diferentes, o que pede padronização para a Regressão Logística (notebook 02).

Em resumo: nesta base, um vinho de alta qualidade tende a ter mais álcool, menos acidez volátil e mais sulfatos, e é essa combinação, e não uma medida isolada, que o modelo vai tentar reconhecer.